# Web Scraper for Sentiment Analysis on Nigerian Governorship Election Candidates

**AI Engineer Candidate — Final Review Presentation**

---

### Agenda

1. **Project Setup** — Clone repo & install dependencies
2. **Run Tests** — 74 unit tests across all modules
3. **Architecture Walkthrough** — Project structure & pipeline flow
4. **Live Pipeline Run** — Scrape → Identify → Analyze → Report
5. **Examine Results** — Candidates, sentiment, profiles
6. **Candidate Identification Deep Dive** — The core innovation
7. **Sentiment Analysis** — Three-tier NLP fallback
8. **Ethical Considerations & Challenges**
9. **Q&A**

---

## 1. Project Setup

Clone the repository and install all required Python packages.

In [ ]:
# Clone the repository
!git clone https://github.com/<your-username>/Web-Scrapper.git
%cd Web-Scrapper

In [ ]:
# Install dependencies (lightweight — VADER handles sentiment, no GPU needed)
!pip install -q requests beautifulsoup4 lxml cloudscraper pyyaml \
    vaderSentiment tweepy praw textblob pandas tqdm aiohttp \
    python-dateutil markdown

---

## 2. Run Tests — Verify Everything Works

**74 unit tests** covering:
- Helper utilities (text cleaning, anonymization, date parsing)
- Candidate identification (extraction, stop names, filtering, deduplication)
- Sentiment analysis (positive/negative/neutral, Pidgin English, batch processing)
- Database operations (save/retrieve, export, deduplication)
- Profile building (themes, excerpts, engagement, demographics)

In [ ]:
!python -m pytest tests/ -v

---

## 3. Architecture Walkthrough

### Project Structure

```
Web-Scrapper/
├── main.py                  # Pipeline orchestrator (entry point)
├── config.yaml              # All configuration (API keys, states, settings)
├── requirements.txt         # Python dependencies
├── report_generator.py      # Markdown report builder
├── scrapers/
│   ├── base.py              # Abstract base scraper class
│   ├── nairaland.py         # Nairaland HTML scraper (BeautifulSoup)
│   ├── reddit.py            # Reddit scraper (PRAW)
│   ├── twitter.py           # Twitter/X scraper (Tweepy API v2)
│   └── facebook.py          # Facebook scraper (Graph API)
├── analysis/
│   ├── candidates.py        # Candidate identification (regex + proximity filtering)
│   ├── sentiment.py         # Sentiment analysis (transformers/VADER/TextBlob)
│   └── profiler.py          # Candidate profile aggregation
├── storage/
│   └── database.py          # SQLite storage + JSON/CSV export
├── utils/
│   ├── config.py            # YAML config loader
│   ├── logger.py            # Logging setup
│   └── helpers.py           # Text cleaning, hashing, date utilities
└── tests/                   # 74 unit tests
```

### Pipeline Flow

```
Scraping → SQLite Storage → Candidate ID → Sentiment Analysis → Profiling → Report
```

### Platform Support

| Platform | Method | Auth | Status |
|----------|--------|------|--------|
| Nairaland | BeautifulSoup HTML parsing | None needed | **Active** |
| Reddit | PRAW (official API) | OAuth credentials | Ready (needs keys) |
| Twitter/X | Tweepy API v2 | Bearer token | Ready (needs keys) |
| Facebook | Graph API (requests) | Access token | Ready (disabled by default) |

> All scrapers inherit from `BaseScraper` and implement `scrape_state(state)`.  
> Platforms with missing API keys are **skipped gracefully** — the pipeline continues with whatever data is available.

Let's look at the configuration that drives the pipeline:

In [ ]:
import yaml

with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

print("States configured:", cfg['states'])
print("Time range:", cfg['general']['time_range_months'], "months")
print("Max posts per platform:", cfg['general']['max_posts_per_platform'])
print("Request delay:", cfg['general']['request_delay'], "seconds")
print("\nSearch keywords:")
for kw in cfg['keywords']:
    print(f"  • {kw}")
print("\nPlatforms:")
for name, settings in cfg['platforms'].items():
    status = 'Enabled' if settings.get('enabled') else 'Disabled'
    print(f"  • {name}: {status}")

---

## 4. Live Pipeline Run

This runs the **full end-to-end pipeline**:

1. **Scrape** Nairaland for Lagos & Kano political discussions (past 12 months)
2. **Store** posts in SQLite with URL-based deduplication
3. **Identify** governorship candidates dynamically from mention frequency
4. **Analyze** sentiment for each post using VADER NLP
5. **Build** detailed candidate profiles
6. **Export** data to JSON/CSV and generate a Markdown report

> **Expected time: ~4-5 minutes** (network-bound; Nairaland scraping with 2s rate-limiting)

In [ ]:
# Remove any stale data from previous runs
import os
db_path = 'output/data/election_data.db'
if os.path.exists(db_path):
    os.remove(db_path)
    print(f"Removed old database: {db_path}")
else:
    print("Clean start — no existing database.")

In [ ]:
# Run the full pipeline with parallel scraping
!python main.py --parallel

---

## 5. Examine Results

### 5.1 Output Files Generated

In [ ]:
import os

output_files = [
    ('Database',         'output/data/election_data.db'),
    ('Scraped Posts (JSON)', 'output/data/scraped_posts.json'),
    ('Scraped Posts (CSV)',  'output/data/scraped_posts.csv'),
    ('Sentiment Results',   'output/data/sentiment_results.json'),
    ('Candidate Profiles',  'output/data/candidate_profiles.json'),
    ('Election Report',     'output/reports/election_report.md'),
]

print(f"{'File':<25} {'Size':>10}  Path")
print("─" * 70)
for label, path in output_files:
    if os.path.exists(path):
        size = os.path.getsize(path)
        unit = 'KB' if size > 1024 else 'B'
        val = size / 1024 if size > 1024 else size
        print(f"{label:<25} {val:>8.1f} {unit}  {path}")
    else:
        print(f"{label:<25} {'MISSING':>10}  {path}")

### 5.2 Candidate Profiles — Who Was Identified?

In [ ]:
import json

with open('output/data/candidate_profiles.json', 'r', encoding='utf-8') as f:
    profiles = json.load(f)

for state, candidates in profiles.items():
    print(f"\n{'='*60}")
    print(f"  {state} STATE — {len(candidates)} candidates identified")
    print(f"{'='*60}")
    print(f"  {'Name':<25} {'Mentions':>8}  {'Sentiment':>10}  Label")
    print(f"  {'─'*55}")
    for c in candidates:
        name = c.get('name', '?')
        mentions = c.get('mention_count', 0)
        sentiment = c.get('sentiment', {})
        overall = sentiment.get('overall', 0)
        if overall > 0.05:
            label = 'Positive'
        elif overall < -0.05:
            label = 'Negative'
        else:
            label = 'Neutral'
        print(f"  {name:<25} {mentions:>8}  {overall:>+10.3f}  {label}")

### 5.3 Sentiment Distribution

In [ ]:
with open('output/data/sentiment_results.json', 'r', encoding='utf-8') as f:
    sentiments = json.load(f)

print(f"Total sentiment entries: {len(sentiments)}\n")

# Group by state
from collections import defaultdict
by_state = defaultdict(lambda: {'pos': 0, 'neg': 0, 'neu': 0, 'total': 0})

for entry in sentiments:
    state = entry.get('state', 'Unknown')
    label = entry.get('label', 'neutral').lower()
    by_state[state]['total'] += 1
    if label == 'positive':
        by_state[state]['pos'] += 1
    elif label == 'negative':
        by_state[state]['neg'] += 1
    else:
        by_state[state]['neu'] += 1

for state, counts in by_state.items():
    t = counts['total']
    print(f"{state}:")
    print(f"  Positive:  {counts['pos']:>4}  ({counts['pos']/t*100:.1f}%)")
    print(f"  Negative:  {counts['neg']:>4}  ({counts['neg']/t*100:.1f}%)")
    print(f"  Neutral:   {counts['neu']:>4}  ({counts['neu']/t*100:.1f}%)")
    print(f"  Total:     {t:>4}\n")

### 5.4 Sample Scraped Posts

In [ ]:
with open('output/data/scraped_posts.json', 'r', encoding='utf-8') as f:
    posts = json.load(f)

print(f"Total posts scraped: {len(posts)}\n")

# Show 3 sample posts
for i, post in enumerate(posts[:3]):
    text = post.get('text', '')[:200]
    print(f"Post {i+1}:")
    print(f"  Platform: {post.get('platform')}")
    print(f"  State:    {post.get('state')}")
    print(f"  Date:     {post.get('date')}")
    print(f"  Text:     {text}...")
    print()

### 5.5 View the Full Report

In [ ]:
from IPython.display import Markdown, display

with open('output/reports/election_report.md', 'r', encoding='utf-8') as f:
    report = f.read()

display(Markdown(report))

---

## 6. Candidate Identification — Deep Dive

### The Challenge

How do you discover governorship candidates from raw forum text **without a hardcoded list?**

### The Solution: Five-Layer Data-Driven Filtering

```
Raw Text
  │
  ▼
[1] Regex Extraction — find 2-3 word capitalized sequences
  │  "Babajide Sanwo-Olu", "Peter Obi", "All Progressives Congress"
  ▼
[2] Stop-Word / Stop-Phrase Filter — remove institutions, parties, states
  │  ✗ "All Progressives Congress"  ✗ "Cross River"  ✗ "National Assembly"
  ▼
[3] Case-Insensitive Recount — forum users write names in lowercase
  │  Regex found 1 match → recount finds 7 actual occurrences
  ▼
[4] State+Gov Proximity Gate (200-char window)
  │  Name + "governor/governorship" + state name must all appear
  │  within 100 chars of each other. Filters out OTHER-STATE governors.
  │  ✗ El-Rufai in Lagos (he's Kaduna governor, just mentioned in Lagos threads)
  ▼
[5] Role Detection (160-char window)
  │  Count: how often is the name near "governor" vs near "president"?
  │  If president_proximity ≥ governor_proximity → national figure → SKIP
  │  ✗ Peter Obi: gov=3, pres=19 → clearly presidential
  │  ✓ Abba Yusuf: gov=14, pres=0 → clearly governorship
  ▼
Final Candidate List (scored & ranked)
```

### Why This Works

The **text itself** reveals each person's political role. No hardcoded politician lists needed.

| Name | Near "governor" | Near "president" | Verdict |
|------|:-:|:-:|:--------|
| Peter Obi | 3 | **19** | Presidential — filtered |
| Bola Tinubu | 3 | **5** | Presidential — filtered |
| Atiku Abubakar | 0 | **10** | Presidential — filtered |
| Abba Yusuf | **14** | 0 | Governorship — **kept** |
| Babajide Sanwo-Olu | **3** | 0 | Governorship — **kept** |
| Ifeanyi Okowa | 0 (in Kano) | — | Wrong state — filtered |

### Let's see the filters in action

We can trace the candidate identification process step by step:

In [ ]:
import sys
sys.path.insert(0, '.')

from storage.database import Database
from analysis.candidates import (
    _extract_names_from_posts, _is_stop_name, _deduplicate_variants,
    _filter_candidates, _GOVERNORSHIP_KEYWORDS, _PRESIDENTIAL_KEYWORDS,
    _ROLE_WINDOW
)

db = Database('output/data/election_data.db')

for state in ['Lagos', 'Kano']:
    posts = db.get_posts_by_state(state)
    print(f"\n{'='*60}")
    print(f"  {state}: {len(posts)} posts")
    print(f"{'='*60}")

    # Step 1: Raw extraction
    raw_names = _extract_names_from_posts(posts)
    print(f"\n  [Step 1] Regex extraction: {len(raw_names)} unique names")

    # Step 2: Deduplication
    deduped = _deduplicate_variants(raw_names)
    print(f"  [Step 2] After deduplication: {len(deduped)} unique names")

    # Step 3: Final filtered candidates
    candidates = _filter_candidates(deduped, posts, state, min_mentions=2)
    print(f"  [Step 3] After proximity + role filtering: {len(candidates)} candidates")
    print(f"\n  Final candidates for {state}:")
    for c in candidates:
        print(f"    • {c['name']}: {c['count']} mentions, score={c['score']:.1f}")

### Demonstrating the Proximity-Based Role Detection

For each name occurrence, we check an 80-char window. The data reveals presidential vs governorship roles:

In [ ]:
# Show how the 80-char proximity window separates presidential from governorship figures
test_names = [
    'peter obi', 'bola tinubu', 'atiku abubakar',
    'babajide sanwo-olu', 'sanwo olu', 'babatunde fashola',
    'abba yusuf', 'umar ganduje', 'rabiu kwankwaso', 'muazu magaji',
]

for state in ['Lagos', 'Kano']:
    posts = db.get_posts_by_state(state)
    print(f"\n{'='*60}")
    print(f"  {state} — Proximity role detection (80-char window)")
    print(f"{'='*60}")
    print(f"  {'Name':<25} {'Gov':>5} {'Pres':>5}  Role")
    print(f"  {'─'*50}")

    for name in test_names:
        gov_prox = 0
        pres_prox = 0
        for p in posts:
            tl = p.get('text', '').lower()
            start = 0
            while True:
                idx = tl.find(name, start)
                if idx == -1:
                    break
                w_start = max(0, idx - _ROLE_WINDOW)
                w_end = min(len(tl), idx + len(name) + _ROLE_WINDOW)
                window = tl[w_start:w_end]
                if any(k in window for k in _GOVERNORSHIP_KEYWORDS):
                    gov_prox += 1
                if any(k in window for k in _PRESIDENTIAL_KEYWORDS):
                    pres_prox += 1
                start = idx + 1
        if gov_prox or pres_prox:
            if gov_prox > pres_prox:
                role = 'GOVERNORSHIP'
            elif pres_prox > gov_prox:
                role = 'PRESIDENTIAL (filtered)'
            else:
                role = 'TIE (filtered)'
            print(f"  {name:<25} {gov_prox:>5} {pres_prox:>5}  {role}")

---

## 7. Sentiment Analysis

### Three-Tier Fallback System

| Tier | Library | Accuracy | Speed | Best For |
|------|---------|----------|-------|----------|
| 1 | HuggingFace Transformers (`twitter-roberta-base`) | High | Slow | Social media text, informal English |
| 2 | **VADER** | Medium | Fast | Rule-based, punctuation-aware |
| 3 | TextBlob | Basic | Fast | Simple polarity detection |

The system tries each tier in order and uses the first available one.  
In this demo, **VADER** is active (no GPU needed, instant startup).

### What Each Candidate Profile Contains

- Overall sentiment score [-1.0 to +1.0] and label
- Positive / Negative / Neutral percentage breakdown
- Per-platform sentiment comparison
- Key discussion themes (frequency-based extraction)
- Top 5 positive and negative excerpts
- Engagement metrics (likes, shares)
- Demographic insights (youth, urban/rural, diaspora indicators)

### Live Sentiment Demo

Let's run the sentiment analyzer on sample texts to see it in action:

In [ ]:
from analysis.sentiment import SentimentAnalyzer

analyzer = SentimentAnalyzer()

sample_texts = [
    "The governor has done excellent work on road infrastructure, very impressed.",
    "This candidate is terrible, nothing but empty promises and corruption.",
    "The new candidate announced his campaign for the 2027 election.",
    "E don try for the people, the governor dey work well.",  # Pidgin English
    "Na wahala for this state, the governor no dey do anything.",  # Pidgin English
]

print(f"Sentiment Analyzer: {analyzer.active_model}\n")
print(f"{'Text (truncated)':<55} {'Score':>7}  Label")
print("─" * 75)

for text in sample_texts:
    result = analyzer.analyze(text)
    display_text = text[:52] + '...' if len(text) > 55 else text
    print(f"{display_text:<55} {result['score']:>+7.3f}  {result['label']}")

### Sentiment Breakdown by Candidate

In [ ]:
for state, candidates in profiles.items():
    print(f"\n{'='*60}")
    print(f"  {state} — Sentiment Breakdown")
    print(f"{'='*60}")
    for c in candidates:
        name = c.get('name', '?')
        s = c.get('sentiment', {})
        overall = s.get('overall', 0)
        pos_pct = s.get('positive_pct', 0)
        neg_pct = s.get('negative_pct', 0)
        neu_pct = s.get('neutral_pct', 0)
        themes = c.get('key_themes', [])[:5]

        print(f"\n  {name}")
        print(f"    Score: {overall:+.3f}  |  +{pos_pct:.0f}%  -{neg_pct:.0f}%  ~{neu_pct:.0f}%")
        if themes:
            theme_str = ', '.join(t if isinstance(t, str) else t[0] for t in themes)
            print(f"    Themes: {theme_str}")

---

## 8. Ethical Considerations & Challenges

### Ethical Safeguards

| Principle | Implementation |
|-----------|---------------|
| **Public data only** | No private accounts, groups, or DMs accessed |
| **Author anonymization** | SHA-256 hash applied before storage — raw usernames never reach the database |
| **Rate limiting** | 2-second delay between requests (configurable) |
| **robots.txt respect** | Checked where applicable |
| **No automated logins** | No credential stuffing or session hijacking |
| **Graceful degradation** | Missing API keys → platform skipped, pipeline continues |

### Challenges Faced & Solutions

| Challenge | Solution |
|-----------|----------|
| **Twitter API costs** ($100/mo for search) | Scraper built & tested; skips gracefully without credentials |
| **Facebook removed public search** (post-Cambridge Analytica) | Page ID-based fetching available; disabled by default |
| **Reddit OAuth approval delay** | PRAW integration complete; skip if no credentials |
| **Nigerian name handling** (hyphens, titles, Pidgin) | Custom regex with stop-word filter, title stripping, case-insensitive recount |
| **Presidential figures appearing as gov candidates** | Proximity-based role detection (80-char window) — fully data-driven |
| **Other-state governors in results** | State+gov proximity gate (200-char window) filters them out |
| **Inconsistent capitalization on Nairaland** | Case-insensitive recount after initial regex extraction |

### Verify: Author Anonymization

In [ ]:
# Show that authors are anonymized in the stored data
with open('output/data/scraped_posts.json', 'r', encoding='utf-8') as f:
    all_posts = json.load(f)

print("Sample stored author fields (SHA-256 hashes — no real usernames):")
seen = set()
for p in all_posts:
    author = p.get('author', '')
    if author and author not in seen and len(seen) < 5:
        seen.add(author)
        print(f"  {author}")
print(f"\nTotal unique anonymized authors: {len(set(p.get('author','') for p in all_posts))}")

---

## 9. Project Stats & Summary

| Metric | Value |
|--------|-------|
| Python modules | 13 |
| Unit tests | 74 (all passing) |
| Platforms supported | 4 (Nairaland, Reddit, Twitter/X, Facebook) |
| States supported | 36 + FCT (configurable) |
| Output formats | SQLite, JSON, CSV, Markdown report |
| Candidate discovery | Fully dynamic — no hardcoded lists |
| Sentiment models | 3-tier fallback (Transformers → VADER → TextBlob) |

### Key Design Strengths

1. **Modular** — Each stage is a separate module; easy to swap components
2. **Configurable** — Single YAML file controls states, platforms, timing, API keys
3. **Resilient** — Graceful degradation when platforms or models are unavailable
4. **Scalable** — ThreadPoolExecutor parallelism; add states by uncommenting config lines
5. **Tested** — 74 unit tests covering all major modules
6. **Ethical** — Anonymization, rate limiting, public data only

---

## Thank You — Questions?

---

### Appendix: Quick Commands

```bash
# Full pipeline (parallel)
python main.py --parallel

# Specific states
python main.py --states Lagos Kano Rivers

# Re-run analysis only (no scraping)
python main.py --skip-scraping

# Run all tests
python -m pytest tests/ -v

# Seed demo data (no network needed)
python seed_demo_data.py
```